# Kharcha final test: is the reader safe to ship?

Run top to bottom on the free Colab CPU (about 15 to 25 minutes, mostly OCR). It checks the **whole reader**
(docTR OCR, trained line model, trained category model, arithmetic confidence) on your 25 bills and ends with a
plain **PASS / FAIL** list.

**How the test stays honest**
* Every score comes from bills the model **never trained on** (4-fold cross-validation, the duplicate photos of one paper bill are kept on the same side).
* The answer key is `data/bills_gold.json`, read by eye from the photos. It can be wrong; the categories marked uncertain are guesses.
* Handwritten bills (1, 21) and cut-off bills (3, 22) are not scored for accuracy. They must simply end up on the **review screen**.
* Only about 20 bills are scorable, so a result moves a few points between runs. Do not quote one number as precise.

| Part | What it does |
|---|---|
| 1 | Install docTR, Tesseract (Nepali) and the Python packages |
| 2 | Upload `reader_server.zip` and `Bills.zip`, run the unit tests |
| 3 | OCR all bills: docTR vs old Tesseract, how many real prices each reads |
| 4 | Label the OCR lines from the answer key |
| 5 | Honest test: rules only vs artificial model vs your trained model, plus the confidence check |
| 6 | PASS / FAIL verdict |
| 7 | Train the final models, check handwritten and cut-off bills go to review |
| 8 | AI off, and the delete-the-AI proof |
| 9 | Try your own photo, then download the models |

## 1. Install

In [ ]:
!apt-get -qq update && apt-get -qq install -y tesseract-ocr tesseract-ocr-eng tesseract-ocr-nep > /dev/null
!pip -q install "python-doctr[torch]" pytesseract opencv-python-headless scikit-learn joblib scipy pillow numpy fastapi uvicorn python-multipart
!tesseract --list-langs

## 2. Upload the code and the bills
Select **both** files in one go: `reader_server.zip` (from this project) and `Bills.zip`.

In [ ]:
import os, sys, glob, shutil, zipfile

from google.colab import files
up = files.upload()
os.makedirs("photos", exist_ok=True)
code_zip = None
for name, data in up.items():
    open(name, "wb").write(data)
    if not name.lower().endswith(".zip"):
        continue
    with zipfile.ZipFile(name) as z:
        images = [m for m in z.namelist() if m.lower().endswith((".jpg", ".jpeg", ".png", ".webp")) and "__MACOSX" not in m]
        if images and not any(m.endswith("parse.py") for m in z.namelist()):
            for m in images:
                with z.open(m) as src, open(os.path.join("photos", os.path.basename(m)), "wb") as dst:
                    shutil.copyfileobj(src, dst)
        else:
            code_zip = name
assert code_zip, "reader_server.zip was not uploaded"
shutil.rmtree("reader_server", ignore_errors=True)
zipfile.ZipFile(code_zip).extractall("tmp_unzip")
root = "tmp_unzip/reader_server" if os.path.isdir("tmp_unzip/reader_server") else "tmp_unzip"
shutil.copytree(root, "reader_server")
shutil.move("photos", "reader_server/photos")
os.chdir("reader_server")
sys.path.insert(0, os.getcwd())
print(len(os.listdir("photos")), "photos;", "answer key:", os.path.exists("data/bills_gold.json"))

In [ ]:
# The reader's own unit tests (fast). All of them must pass before anything below means anything.
r = !python -m unittest discover 2>&1 | tail -4
print("\n".join(r))

## 3. OCR: docTR (new) vs Tesseract (old)
"Prices found" = how many of the real item prices and totals (from the answer key) appear as numbers in the OCR text.
Tesseract scored about 71% and docTR about 93% on your earlier test; this repeats it with the exact code the reader uses.

In [ ]:
import json, time
from PIL import Image
import ocr, bills

ALLOW_TESSERACT = os.environ.get("KHARCHA_DRY_RUN") == "1"        # only for testing this notebook without docTR
assert ocr.doctr_available() or ALLOW_TESSERACT, "docTR did not install. Run the install cell again (Runtime > Restart session if needed)."
print("OCR engine:", "docTR (Tesseract is the Nepali / rescue path)" if ocr.doctr_available() else "Tesseract only (DRY RUN)")

gold = bills.load_gold()
names = sorted((n for n in gold if os.path.exists(f"photos/{n}")), key=lambda n: (len(n), n))
ocr_new, ocr_old = {}, {}
t0 = time.time()
for n in names:
    img = Image.open(f"photos/{n}")
    ocr_new[n] = [l.strip() for l in ocr.ocr_lines(img) if l.strip()]
print(f"reader OCR on {len(names)} photos: {time.time() - t0:.0f} s")
t0 = time.time()
for n in names:
    ocr_old[n] = [l.strip() for l in ocr._tesseract_lines(Image.open(f"photos/{n}")) if l.strip()]
print(f"Tesseract only: {time.time() - t0:.0f} s")

print()
print(f"{'bill':8}{'reader OCR':>12}{'Tesseract':>12}")
tot = {"new": [0, 0], "old": [0, 0]}
for n in names:
    if not gold[n]["scorable"]:
        continue
    a, b = bills.price_recall(ocr_new[n], gold[n]), bills.price_recall(ocr_old[n], gold[n])
    for k, v in (("new", a), ("old", b)):
        tot[k][0] += v[0]; tot[k][1] += v[1]
    print(f"{n:8}{a[0]:>8}/{a[1]:<3}{b[0]:>8}/{b[1]:<3}")
print(f"\nprices + totals found:  reader OCR {tot['new'][0]}/{tot['new'][1]} = {100 * tot['new'][0] / tot['new'][1]:.0f}%    Tesseract {tot['old'][0]}/{tot['old'][1]} = {100 * tot['old'][0] / tot['old'][1]:.0f}%")
json.dump({"reader_ocr": ocr_new, "tesseract": ocr_old}, open("ocr_lines.json", "w"), ensure_ascii=False)

In [ ]:
# Look at one bill the way the models see it
for i, l in enumerate(ocr_new["5.jpg"]):
    print(f"[{i:2}] {l}")

## 4. Label the lines from the answer key
Each OCR line is marked merchant / date / item / subtotal / tax / total / other by matching it to the answer key.
This is how the line model learns from your bills. A few lines will be mislabeled where the OCR garbled them; that is normal and the test accounts for it.

In [ ]:
from collections import Counter
rec = bills.build_receipts(ocr_new, gold)
print(len(rec), "bills labelled;", len(bills.trainable(rec)), "are used for training and scoring")
print(Counter(l for r in rec.values() for _, l in r["lines"]))
# a bill where the answer key says there are items, but no OCR line matched any: the OCR or the key is wrong there
for n in bills.trainable(rec):
    labs = [l for _, l in rec[n]["lines"]]
    want = len(gold[n]["items"])
    if labs.count("item") < want or "total" not in labs:
        print(f"  {n}: key has {want} items, labelled {labs.count('item')}; total line {'found' if 'total' in labs else 'NOT found'}")

## 5. The honest test
For each bill, a line model trained on the *other* bills reads it. Three systems are compared:
* **rules**: no trained model, only regular expressions (the baseline the AI has to beat)
* **artificial**: trained on generated fake receipts only
* **mine**: fake receipts + your other bills

Confidence counts a read as trustworthy only when the **arithmetic holds** (items + VAT - discount = total, or cash - change = total).

In [ ]:
res = bills.cross_validate(rec, gold, k=4, seed=0, synthetic=3000)
bills.report(res)

In [ ]:
# Same test on the old Tesseract text, so you can see what the OCR change is worth
rec_old = bills.build_receipts(ocr_old, gold)
res_old = bills.cross_validate(rec_old, gold, k=4, seed=0, synthetic=3000)
print("OLD OCR (Tesseract only)")
bills.report(res_old)

In [ ]:
# Stability: other random splits (the numbers move a few points; if they swing wildly, do not trust any single run)
for seed in (1, 2):
    r = bills.cross_validate(rec, gold, k=4, seed=seed, synthetic=3000)
    m = [x["mine"] for x in r]
    print(f"seed {seed}: total right {100*sum(x['total'] for x in m)/len(m):.0f}%, all items right {100*sum(x['items'] for x in m)/len(m):.0f}%, "
          f"wrong totals that cleared 0.7: {sum((not x['total']) and x['confidence']>=0.7 for x in m)}")

## 6. Verdict

In [ ]:
checks = bills.verdict(res)
for name, ok, detail in checks:
    print(("PASS  " if ok else "FAIL  ") + name + "   [" + detail + "]")
print()
if all(ok for _, ok, _ in checks):
    print("All checks pass on held-out bills. Remember this is ~20 bills: it shows the design works, it is not a guarantee.")
else:
    print("Not ready. Read the per-bill table above: bills marked WRONG with a high confidence are the ones to look at first.")

## 7. Train the final models, then check the hard bills
Trains `mine` on every scorable bill. The hard bills below were **not** used for training: handwritten (1, 24, 21) and cut off (3, 22).
They must all end up needing review.

In [ ]:
from label_tools import item_categories
from train import train
tr = bills.trainable(rec)
train("mine", 4000, None, [rec[n]["lines"] for n in tr], item_categories({n: rec[n] for n in tr}), "artificial")
from pipeline import read_lines, load_models
MINE = load_models("mine")
print()
print(f"{'bill':8}{'conf':>6}  {'review':7} total   why")
for n in ("1.jpg", "24.jpg", "21.jpg", "3.jpg", "22.jpg"):
    if n in ocr_new:
        r = read_lines(ocr_new[n], MINE)
        print(f"{n:8}{r['confidence']:>6.2f}  {str(r['needs_review']):7} {str(r['total']):>7}  {r['reconciliation']}")

## 8. AI off, and the delete-the-AI proof (for the hackathon demo)

In [ ]:
from pipeline import read_photo, ModelsMissing

def show(r):
    print(f"  merchant={r.get('merchant')}  date={r.get('date')}{' (approx)' if r.get('date_approx') else ''}  total={r.get('total')}  confidence={r.get('confidence')}  needs_review={r.get('needs_review')}")
    print(f"  {r.get('reconciliation')}")
    for i in r["items"]:
        print(f"    {i['name'][:34]:34} x{i.get('qty',1)} {i['price']:>9.2f}  {i.get('category')}")

demo = "photos/15.jpg"
print("### AI ON (trained models)"); show(read_photo(demo, models="mine"))
print("\n### AI OFF (rules only, same OCR)"); show(read_photo(demo, models=None))
print("\n### MODEL FILES DELETED")
shutil.move("models", "models_hidden")
try:
    read_photo(demo, models="mine"); print("It still read the receipt: this must not happen.")
except ModelsMissing as e:
    print("Refused, as it should:", e)
finally:
    shutil.move("models_hidden", "models")

## 9. Try your own photo, then download
Use a bill that is **not** in Bills.zip. If it says NEEDS REVIEW, that is the app sending it to the review screen: check whether that was the right call.

In [ ]:
import io
from IPython.display import display
up = files.upload()
for name, data in up.items():
    img = Image.open(io.BytesIO(data)); img.load()
    t = img.copy(); t.thumbnail((450, 450)); display(t)
    print("=" * 60, name)
    r = read_photo(img, models="mine")
    print("NEEDS REVIEW" if r["needs_review"] else "AUTO-APPROVED")
    show(r)

In [ ]:
# Everything to take back: trained models, the per-bill results of the honest test, the OCR text
import json
slim = [{"bill": r["bill"], **{s: {k: v for k, v in r[s].items() if k != "pred"} for s in ("rules", "artificial", "mine") if s in r}} for r in res]
json.dump(slim, open("final_test_results.json", "w"), indent=1)
shutil.make_archive("kharcha_models", "zip", "models")
files.download("kharcha_models.zip"); files.download("final_test_results.json"); files.download("ocr_lines.json")